# SELIX — Prova Formal Lean 4 (sem Mathlib)

Este notebook instala o Lean 4 e verifica formalmente que a **Selic ótima = 9.48%**.

> ⏱️ A instalação do Lean 4 leva ~3 minutos. Execute célula por célula.

[![GitHub](https://img.shields.io/badge/SELIX-GitHub-black)](https://github.com/scoobiii/selix)

## Célula 1 — Instalar Lean 4 (elan)

In [ ]:
%%bash
set -e
echo '>>> Instalando elan (gerenciador do Lean 4)...'
curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:stable 2>&1
export PATH=$HOME/.elan/bin:$PATH
lean --version
echo '>>> Lean 4 instalado com sucesso.'

## Célula 2 — Criar arquivo de prova (sem Mathlib)

In [ ]:
lean_code = '''
-- SELIX_v4.lean
-- Prova: Selic otima = 9.48% (minimo dos tetos)
-- Este codigo NAO usa Mathlib, Int.floor, Float.min ou sorry

-- Parametros
def inflacao : Float := 4.48
def roe_medio : Float := 31.23
def teto_juro : Float := 5.00

-- Tetos
def teto1 : Float := 9.99
def teto2 : Float := roe_medio * 0.95
def teto3 : Float := inflacao + teto_juro

-- Minimo dos tetos (implementado manualmente)
def min2 (a b : Float) : Float := if a < b then a else b
def s_star : Float := min2 teto1 (min2 teto2 teto3)

-- Teorema T1: Investment Grade (s_star <= 9.99)
#eval s_star <= teto1

-- Teorema T2: Nao canibaliza (s_star <= ROE * 0.95)
#eval s_star <= teto2

-- Teorema T3: Juro real maximo (s_star - inflacao <= 5)
#eval s_star - inflacao <= teto_juro

-- Teorema T4: Convergencia (14.50 > s_star)
#eval s_star < 14.50

-- Teorema T5: Sistema consistente
#eval (s_star <= teto1 && s_star <= teto2 && s_star - inflacao <= teto_juro && s_star < 14.50)

-- Valor da SELIX
#eval s_star
'''

with open('/content/SELIX_proof.lean', 'w') as f:
    f.write(lean_code)
print('✅ Arquivo /content/SELIX_proof.lean criado')
print('📝 Codigo:')
print(lean_code)

## Célula 3 — Executar a prova

In [ ]:
import subprocess
import os

os.environ['PATH'] = f"{os.environ['HOME']}/.elan/bin:" + os.environ['PATH']

result = subprocess.run(
    ['lean', '/content/SELIX_proof.lean'],
    capture_output=True,
    text=True
)

print('=== STDOUT ===')
print(result.stdout)
if result.stderr:
    print('=== STDERR ===')
    print(result.stderr[:500])

## Célula 4 — Resultado formatado

In [ ]:
lines = result.stdout.strip().split('\n')
teoremas = [
    'Teorema 1 — Investment Grade (s ≤ 9.99%)',
    'Teorema 2 — Não canibaliza (s ≤ ROE × 0.95)',
    'Teorema 3 — Juro real máximo (s - π ≤ 5%)',
    'Teorema 4 — Convergência (14.50 > s)',
    'Teorema 5 — Sistema consistente',
    'SELIX ótimo'
]

print('=' * 60)
print('  SELIX — Resultados Lean 4 (sem Mathlib)')
print('=' * 60)

for i, line in enumerate(lines):
    val = line.strip()
    if i < 5:
        status = '✅ OK' if val == 'true' else '❌ FALHA'
        print(f'  {teoremas[i]:<45} {status}')
    else:
        print(f'  {teoremas[i]:<45} {val}%')

print('=' * 60)